# Cycle 1 — Modelling: Match Outcome Prediction (Win / Draw / Loss)

**Project:** Football Predictor  
**Inputs:** `data/processed/premier_league_matches_processed.csv` and `data/processed/skysports_match_stats_processed.csv`  
**Depends on:** All exploration, preprocessing, and feature engineering notebooks

---

## Purpose of this Notebook

This notebook trains and evaluates machine learning models for predicting Premier League match outcomes (Home Win / Draw / Away Win). Models are trained on **both** processed datasets separately so we can compare results.


## Model Progression

We train 4 models in increasing complexity:

| Model | Why we use it |
|---|---|
| **Dummy Classifier** | Always predicts the most common class (Home Win). Sets the floor — any real model must beat this |
| **Logistic Regression** | Simple, interpretable linear model. Fast baseline for structured data |
| **Random Forest** | Ensemble of decision trees. Handles non-linear patterns, robust to noise |
| **XGBoost** | Gradient boosting. Generally highest accuracy for tabular data |

In [1]:
import sys, os

# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing

---
## Cell 1 — Imports

**What it does:** Imports all required libraries.

**Why:** Keeping all imports at the top makes it clear what the notebook depends on.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

print('All libraries imported successfully')

All libraries imported successfully


---
# PART A — Dataset 1: `premier_league_matches_processed.csv`

**6,840 rows | 34 features | 18 seasons (2000–2018)**

Features: season totals, points per game, last 5 results, form points, streaks, goal difference, team identity, matchweek, season.

---
## A1 — Load and Prepare Data

**What it does:** Loads the processed dataset, separates features (X) from target (y), and splits into train/test sets.

**Why:** The 80/20 train/test split is standard. `random_state=42` ensures reproducibility — running this again gives the same split.

In [3]:
df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED))

X1 = df1.drop(columns=['FTR', 'Season'])  # Season is meta (year), not a feature
y1 = df1['FTR']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

# Scale features for Logistic Regression
scaler1 = StandardScaler()
X1_train_s = scaler1.fit_transform(X1_train)
X1_test_s  = scaler1.transform(X1_test)

print('Total rows:', len(df1))
print('Features:', X1.shape[1])
print('Training rows:', len(X1_train))
print('Test rows:', len(X1_test))
print()
print('Target distribution (full dataset):')
print(y1.value_counts().sort_index())
print('(0=Away Win, 1=Draw, 2=Home Win)')

Total rows: 6840
Features: 33
Training rows: 5472
Test rows: 1368

Target distribution (full dataset):
FTR
0    1913
1    1751
2    3176
Name: count, dtype: int64
(0=Away Win, 1=Draw, 2=Home Win)


### Observations
- Class imbalance: Home wins are almost twice as common as draws
- A dummy model always predicting Home Win would score ~46.4% — this is the floor to beat
- StandardScaler fitted on training data only, then applied to test — prevents data leakage from scaling

---
## A2 — Model 1: Dummy Classifier

**What it does:** Always predicts the most frequent class (Home Win). No learning involved.

**Why:** Establishes the absolute minimum baseline. If a real model cannot beat this, it has learned nothing useful.

**Comparison with FinalYearProject:** Same model used. FYP Dummy got ~38.55% (on a different, smaller dataset). Here the baseline is higher (~46%) because Home Wins are more frequent in this larger dataset.

In [4]:
dummy1 = DummyClassifier(strategy='most_frequent', random_state=42)
dummy1.fit(X1_train, y1_train)
y_pred_dummy1 = dummy1.predict(X1_test)

print('DUMMY CLASSIFIER — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_dummy1):.4f} ({accuracy_score(y1_test, y_pred_dummy1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_dummy1, target_names=['Away Win', 'Draw', 'Home Win']))

DUMMY CLASSIFIER — Dataset 1
Accuracy: 0.4635 (46.35%)

              precision    recall  f1-score   support

    Away Win       0.00      0.00      0.00       394
        Draw       0.00      0.00      0.00       340
    Home Win       0.46      1.00      0.63       634

    accuracy                           0.46      1368
   macro avg       0.15      0.33      0.21      1368
weighted avg       0.21      0.46      0.29      1368



### Observations
- Accuracy: **46.35%** — this is the floor
- Predicts Home Win for every match — recall 1.00 for Home Win, 0 for the other classes
- Away Win and Draw are completely ignored — precision and recall both 0
- This reflects the class imbalance: Home Win (46.4% of training data) is the safe default guess

### Notes for Report
- A baseline of 46.35% means we need to significantly exceed this to claim the model is learning anything
- Draw prediction is the hardest class — all models struggle here

---
## A3 — Model 2: Logistic Regression

**What it does:** Fits a linear decision boundary between classes. Simple, fast, and interpretable.

**Why:** The first real ML model in the progression. If Logistic Regression already beats the dummy by a meaningful margin, the features contain real signal.

**`class_weight='balanced'`:** Adjusts for class imbalance automatically — gives more weight to minority classes (Draw, Away Win) during training.

**Comparison with FinalYearProject:** FYP Logistic Regression got 46.18% on the leakage-contaminated dataset. Here we get a clean number.

In [5]:
lr1 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr1.fit(X1_train_s, y1_train)
y_pred_lr1 = lr1.predict(X1_test_s)

print('LOGISTIC REGRESSION — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_lr1):.4f} ({accuracy_score(y1_test, y_pred_lr1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_lr1, target_names=['Away Win', 'Draw', 'Home Win']))

LOGISTIC REGRESSION — Dataset 1
Accuracy: 0.4985 (49.85%)

              precision    recall  f1-score   support

    Away Win       0.46      0.59      0.52       394
        Draw       0.30      0.25      0.27       340
    Home Win       0.64      0.57      0.60       634

    accuracy                           0.50      1368
   macro avg       0.46      0.47      0.46      1368
weighted avg       0.50      0.50      0.50      1368



### Observations
- Accuracy: **49.85%** — beats the dummy by **3.50 percentage points**
- Now predicting all 3 classes — the model is actually learning
- Home Win: best precision (0.64) — model is fairly confident when it predicts a home win
- Draw: worst performance (f1 = 0.27) — draws are genuinely difficult to predict
- Away Win: good recall (0.59) — catches roughly 60% of actual away wins

### Notes for Report
- Logistic Regression beats the dummy, confirming the features have predictive signal
- Draw prediction difficulty is a known challenge in football analytics

---
## A4 — Model 3: Random Forest

**What it does:** Trains 100 decision trees on random subsets of data and features, then aggregates their predictions.

**Why:** Handles non-linear relationships that Logistic Regression cannot. More powerful than a single decision tree because averaging 100 trees reduces overfitting.

**Comparison with FinalYearProject:** FYP Random Forest Tuned got 49.87%. Here we get a clean comparable number.

In [6]:
rf1 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf1.fit(X1_train, y1_train)
y_pred_rf1 = rf1.predict(X1_test)

print('RANDOM FOREST — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_rf1):.4f} ({accuracy_score(y1_test, y_pred_rf1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_rf1, target_names=['Away Win', 'Draw', 'Home Win']))

RANDOM FOREST — Dataset 1
Accuracy: 0.5139 (51.39%)

              precision    recall  f1-score   support

    Away Win       0.50      0.42      0.45       394
        Draw       0.28      0.10      0.14       340
    Home Win       0.55      0.80      0.65       634

    accuracy                           0.51      1368
   macro avg       0.44      0.44      0.42      1368
weighted avg       0.47      0.51      0.47      1368



### Observations
- Accuracy: **51.39%** — beats Logistic Regression by **1.54 percentage points**
- Home Win recall is high (0.80) — very good at identifying home wins
- Draw recall drops to **0.10** — the model barely predicts draws at all
- Random Forest leans towards the majority class despite `balanced` weights — the gain over LR comes from majority-class precision, not from learning draws

### Notes for Report
- The progression Dummy → LogReg → RF (46% → 50% → 51%) shows consistent improvement
- Draw prediction remains the weakest point — a pattern across all models

---
## A5 — Model 4: XGBoost

**What it does:** Gradient boosting — builds trees sequentially, each one correcting the errors of the previous. Generally the strongest performer on tabular data.

**Why:** The most powerful model in the progression. In FinalYearProject, XGBoost Tuned achieved 50.92% — but on leakage-contaminated data. Here we see what XGBoost achieves on clean data.

**Comparison with FinalYearProject:** FYP XGBoost Tuned = 50.92% (invalid — leakage). Here we get the honest equivalent.

In [7]:
xgb1 = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', verbosity=0)
xgb1.fit(X1_train, y1_train)
y_pred_xgb1 = xgb1.predict(X1_test)

print('XGBOOST — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_xgb1):.4f} ({accuracy_score(y1_test, y_pred_xgb1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_xgb1, target_names=['Away Win', 'Draw', 'Home Win']))

XGBOOST — Dataset 1
Accuracy: 0.5095 (50.95%)

              precision    recall  f1-score   support

    Away Win       0.47      0.43      0.45       394
        Draw       0.34      0.20      0.25       340
    Home Win       0.57      0.73      0.64       634

    accuracy                           0.51      1368
   macro avg       0.46      0.45      0.45      1368
weighted avg       0.48      0.51      0.49      1368



### Observations
- Accuracy: **50.95%** — slightly below Random Forest (51.39%) without tuning
- Better draw prediction than Random Forest (recall **0.20 vs 0.10**)
- XGBoost is more balanced across all three classes
- With hyperparameter tuning, XGBoost typically surpasses Random Forest

### Notes for Report
- XGBoost without tuning is not always the best — it needs tuning to shine
- The untuned XGBoost still beats the dummy by ~4.6 percentage points
- Tuning is the natural next step (Cycle 2)

---
## A6 — Dataset 1 Results Summary

**What it does:** Summarises all model results on Dataset 1 in one table.

**Why:** Easy to compare all models at a glance and identify the best performer.

In [8]:
results_d1 = pd.DataFrame({
    'Model': ['Dummy Classifier', 'Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y1_test, y_pred_dummy1),
        accuracy_score(y1_test, y_pred_lr1),
        accuracy_score(y1_test, y_pred_rf1),
        accuracy_score(y1_test, y_pred_xgb1)
    ]
})
results_d1['Accuracy %'] = (results_d1['Accuracy'] * 100).round(2)
results_d1['vs Dummy'] = ((results_d1['Accuracy'] - results_d1['Accuracy'].iloc[0]) * 100).round(2)
print('Dataset 1 — premier_league_matches_processed')
print(results_d1.to_string(index=False))

Dataset 1 — premier_league_matches_processed
              Model  Accuracy  Accuracy %  vs Dummy
   Dummy Classifier  0.463450       46.35      0.00
Logistic Regression  0.498538       49.85      3.51
      Random Forest  0.513889       51.39      5.04
            XGBoost  0.509503       50.95      4.61


### Observations
- **Random Forest is the best untuned model on Dataset 1 at 51.39%**
- LR → RF improves accuracy (49.85 → 51.39), but XGBoost (50.95) actually under-performs RF without tuning. So the progression isn't strictly monotonic on this dataset
- XGBoost is close behind RF and is likely to surpass it with hyperparameter tuning (Cycle 2 of work)
- All four models beat the dummy by 4.6–5.0pp, confirming the features carry real predictive signal

---
# PART B — Dataset 2: `skysports_match_stats_processed.csv`

**1,123 rows | 19 features | 3 seasons (2020–2023)**

Features: rolling averages of possession, shots, shots on target, pass accuracy, tackles, corners, fouls, yellow cards — for both home and away teams — over the last 5 matches.

---
## B1 — Load and Prepare Data

**What it does:** Loads Dataset 2, drops the `date` column (not a feature), and prepares train/test split.

**Why:** `date` was kept in the processed file for reference but is not a predictive feature — we drop it now before training.

In [9]:
df2 = pd.read_csv(str(Paths.SKYSPORTS_PROCESSED))

X2 = df2.drop(columns=['FTR', 'date'])
y2 = df2['FTR']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s  = scaler2.transform(X2_test)

print('Total rows:', len(df2))
print('Features:', X2.shape[1])
print('Training rows:', len(X2_train))
print('Test rows:', len(X2_test))
print()
print('Target distribution:')
print(y2.value_counts().sort_index())
print('(0=Away Win, 1=Draw, 2=Home Win)')

Total rows: 1121
Features: 19
Training rows: 896
Test rows: 225

Target distribution:
FTR
0    380
1    256
2    485
Name: count, dtype: int64
(0=Away Win, 1=Draw, 2=Home Win)


### Observations
- Significantly fewer rows (**1,121 vs 6,840**) — smaller test set (225) means results are noisier
- Class distribution: 33.9% Away / 22.8% Draw / 43.3% Home. Home advantage is similar to Dataset 1, but the **draw class is rarer** here (22.8% vs 25.6%) — drawing detection will be harder
- **19 features vs 33** (Dataset 1 dropped Season as meta) — fewer but rolling match stats (possession, shots, tactics) carry richer per-match signal

---
## B2 — Dummy Classifier

In [10]:
dummy2 = DummyClassifier(strategy='most_frequent', random_state=42)
dummy2.fit(X2_train, y2_train)
y_pred_dummy2 = dummy2.predict(X2_test)

print('DUMMY CLASSIFIER — Dataset 2')
print(f'Accuracy: {accuracy_score(y2_test, y_pred_dummy2):.4f} ({accuracy_score(y2_test, y_pred_dummy2)*100:.2f}%)')
print()
print(classification_report(y2_test, y_pred_dummy2, target_names=['Away Win', 'Draw', 'Home Win']))

DUMMY CLASSIFIER — Dataset 2
Accuracy: 0.3644 (36.44%)

              precision    recall  f1-score   support

    Away Win       0.00      0.00      0.00        80
        Draw       0.00      0.00      0.00        63
    Home Win       0.36      1.00      0.53        82

    accuracy                           0.36       225
   macro avg       0.12      0.33      0.18       225
weighted avg       0.13      0.36      0.19       225



### Observations
- Baseline is **36.44%** — lower than Dataset 1 (46.35%) because Home Wins are less dominant in this dataset
- The lower baseline means there is more room for models to demonstrate improvement

---
## B3 — Logistic Regression

In [11]:
lr2 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr2.fit(X2_train_s, y2_train)
y_pred_lr2 = lr2.predict(X2_test_s)

print('LOGISTIC REGRESSION — Dataset 2')
print(f'Accuracy: {accuracy_score(y2_test, y_pred_lr2):.4f} ({accuracy_score(y2_test, y_pred_lr2)*100:.2f}%)')
print()
print(classification_report(y2_test, y_pred_lr2, target_names=['Away Win', 'Draw', 'Home Win']))

LOGISTIC REGRESSION — Dataset 2
Accuracy: 0.4489 (44.89%)

              precision    recall  f1-score   support

    Away Win       0.47      0.53      0.50        80
        Draw       0.22      0.17      0.20        63
    Home Win       0.55      0.59      0.57        82

    accuracy                           0.45       225
   macro avg       0.42      0.43      0.42       225
weighted avg       0.43      0.45      0.44       225



### Observations
- Accuracy: **44.89%** — beats the dummy by **+8.45 percentage points** — more than 2× the LR gain on Dataset 1 (+3.50pp)
- All three classes are predicted with non-zero precision and recall — better balance than Dataset 1's RF/XGB which collapsed the draw class
- Draw recall is **0.17** — lower than Dataset 1's LR (0.25), reflecting the smaller absolute count of draws (~256 total) and the smaller test set (225 rows)
- The rolling features (possession, shots, tackles) carry strong signal — the per-match resolution beats Dataset 1's season-level form stats relative to baseline

### Notes for Report
- The larger gap over the dummy (**+8.45pp vs +3.50pp**) suggests rolling match statistics contain stronger signal than season-level form features

---
## B4 — Random Forest

In [12]:
rf2 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf2.fit(X2_train, y2_train)
y_pred_rf2 = rf2.predict(X2_test)

print('RANDOM FOREST — Dataset 2')
print(f'Accuracy: {accuracy_score(y2_test, y_pred_rf2):.4f} ({accuracy_score(y2_test, y_pred_rf2)*100:.2f}%)')
print()
print(classification_report(y2_test, y_pred_rf2, target_names=['Away Win', 'Draw', 'Home Win']))

RANDOM FOREST — Dataset 2
Accuracy: 0.4622 (46.22%)

              precision    recall  f1-score   support

    Away Win       0.51      0.51      0.51        80
        Draw       0.25      0.03      0.06        63
    Home Win       0.45      0.74      0.56        82

    accuracy                           0.46       225
   macro avg       0.40      0.43      0.38       225
weighted avg       0.41      0.46      0.40       225



### Observations
- Accuracy: **46.22%** — beats Logistic Regression by 1.33pp on raw accuracy, but the win is **majority-class biased**
- RF effectively collapses the draw class (recall **0.03**, f1 = 0.06) — almost never predicts draws — whereas Logistic Regression predicts draws meaningfully (recall 0.17, f1 = 0.20)
- This shows that accuracy alone does not tell the full story — depending on whether "draws matter" for the use case, **Logistic Regression may be the preferred model here despite lower accuracy**

### Notes for Report
- Two models with similar headline accuracy can have very different per-class behaviour
- For football prediction where draw detection matters, Logistic Regression is more useful than Random Forest on this datasetst on this dataset

---
## B5 — XGBoost

In [13]:
xgb2 = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', verbosity=0)
xgb2.fit(X2_train, y2_train)
y_pred_xgb2 = xgb2.predict(X2_test)

print('XGBOOST — Dataset 2')
print(f'Accuracy: {accuracy_score(y2_test, y_pred_xgb2):.4f} ({accuracy_score(y2_test, y_pred_xgb2)*100:.2f}%)')
print()
print(classification_report(y2_test, y_pred_xgb2, target_names=['Away Win', 'Draw', 'Home Win']))

XGBOOST — Dataset 2
Accuracy: 0.4222 (42.22%)

              precision    recall  f1-score   support

    Away Win       0.41      0.40      0.41        80
        Draw       0.24      0.10      0.14        63
    Home Win       0.47      0.70      0.56        82

    accuracy                           0.42       225
   macro avg       0.37      0.40      0.37       225
weighted avg       0.38      0.42      0.39       225



### Observations
- Accuracy: **42.22%** — the **worst-performing model** on this dataset, below both Logistic Regression (44.89%) and Random Forest (46.22%)
- XGBoost underperforms on smaller datasets without tuning — it tends to overfit when training data is limited (only 896 training rows here)
- With hyperparameter tuning (fewer trees, stronger regularisation, lower learning rate), XGBoost typically improves significantly — this happens in `cycle1_tuning.ipynb`

### Notes for Report
- XGBoost needs both more data and tuning to reach its potential — out-of-the-box defaults overfit on ~1k-row datasets
- On Dataset 2 untuned, Logistic Regression is surprisingly competitive — a textbook example of a simple model outperforming a complex one on small datasets

---
## B6 — Dataset 2 Results Summary

In [14]:
results_d2 = pd.DataFrame({
    'Model': ['Dummy Classifier', 'Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y2_test, y_pred_dummy2),
        accuracy_score(y2_test, y_pred_lr2),
        accuracy_score(y2_test, y_pred_rf2),
        accuracy_score(y2_test, y_pred_xgb2)
    ]
})
results_d2['Accuracy %'] = (results_d2['Accuracy'] * 100).round(2)
results_d2['vs Dummy'] = ((results_d2['Accuracy'] - results_d2['Accuracy'].iloc[0]) * 100).round(2)
print('Dataset 2 — skysports_match_stats_processed')
print(results_d2.to_string(index=False))

Dataset 2 — skysports_match_stats_processed
              Model  Accuracy  Accuracy %  vs Dummy
   Dummy Classifier  0.364444       36.44      0.00
Logistic Regression  0.448889       44.89      8.44
      Random Forest  0.462222       46.22      9.78
            XGBoost  0.422222       42.22      5.78


---
# PART C — Full Comparison

**What it does:** Compares all models across both datasets in a single table, and draws conclusions about which dataset and model to take forward.


In [15]:
comparison = pd.DataFrame({
    'Model': ['Dummy', 'Logistic Regression', 'Random Forest', 'XGBoost'],
    'Dataset 1 (6,840 rows)': [
        f'{accuracy_score(y1_test, y_pred_dummy1)*100:.2f}%',
        f'{accuracy_score(y1_test, y_pred_lr1)*100:.2f}%',
        f'{accuracy_score(y1_test, y_pred_rf1)*100:.2f}%',
        f'{accuracy_score(y1_test, y_pred_xgb1)*100:.2f}%'
    ],
    'Dataset 2 (1,121 rows)': [
        f'{accuracy_score(y2_test, y_pred_dummy2)*100:.2f}%',
        f'{accuracy_score(y2_test, y_pred_lr2)*100:.2f}%',
        f'{accuracy_score(y2_test, y_pred_rf2)*100:.2f}%',
        f'{accuracy_score(y2_test, y_pred_xgb2)*100:.2f}%'
    ],
})
print(comparison.to_string(index=False))

              Model Dataset 1 (6,840 rows) Dataset 2 (1,121 rows)
              Dummy                 46.35%                 36.44%
Logistic Regression                 49.85%                 44.89%
      Random Forest                 51.39%                 46.22%
            XGBoost                 50.95%                 42.22%


## Key Conclusions

### 1. Dataset 1 produces higher absolute accuracy; Dataset 2 extracts more signal per feature

**Best untuned models:**
- Dataset 1: Random Forest at **51.39%** (baseline 46.35% → gain **+5.04pp**)
- Dataset 2: Random Forest at **46.22%** (baseline 36.44% → gain **+9.78pp**)

Dataset 1 wins on raw accuracy. But Dataset 2 has a much lower dummy baseline (Home Wins are less dominant in 2020-23 PL than they were across the 2000-2017 PL), so Dataset 2's models extract roughly **twice the signal over baseline** that Dataset 1's models extract. The feature set is also smaller (19 vs 33 cols) — denser predictive value per feature.

### 2. Draw prediction remains the hardest problem

Across both datasets, every model collapses to near-zero recall on the Draw class except Logistic Regression. Random Forest's apparent accuracy gain over LR is largely explained by it ignoring draws and over-predicting the majority class. This is a well-documented challenge in football analytics — draws are inherently noisy outcomes.

### 3. Logistic Regression is the only model that meaningfully predicts draws

On Dataset 2, Logistic Regression is the only model that keeps a non-zero draw recall (**0.17**, vs 0.03 for Random Forest and 0.10 for XGBoost). On Dataset 1, LR also has the best draw recall (**0.25** vs 0.10 RF, 0.20 XGB). If the downstream use case cares about draw detection, **Logistic Regression is the preferred Cycle 1 baseline despite trailing on raw accuracy**.

### 4. XGBoost needs tuning to be competitive

Untuned XGBoost finishes **last on Dataset 2 (42.22%)** — below LR and RF. On Dataset 1 it lands between LR and RF (50.95%). The defaults overfit on small datasets and rely on hyperparameter tuning to regularize. This is what `cycle1_tuning.ipynb` addresses; the deployed Cycle 1 model is the tuned XGBoost.

---
## Decision: Which dataset to take forward?

**Dataset 2 (Sky Sports rolling features) is taken forward to tuning and deployment**, despite its lower raw accuracy. The reasons:

- **Larger gain over baseline** (+9.78pp vs +5.04pp) means features carry more predictive signal per match
- **Per-match resolution** (rolling possession, shots, tackles) is more relevant for predicting an upcoming specific match than season-level form
- **Same-team coverage**: Dataset 2's 25 teams overlap with the current PL, so the feature store stays useful for production predictions; Dataset 1 covers historical PL teams that no longer exist in the league
- **Tuning headroom**: untuned numbers don't show the ceiling. Cycle 1 tuning lifts XGBoost on Dataset 2 to **57.33%** under random split, well above any untuned Dataset 1 model

**Dataset 1 is kept as a comparative baseline** showing the full PL history but is not used for deployment.

**Important caveat:** these comparisons use a random train/test split. The chronological-split notebooks in `notebooks_chronological/` re-evaluate the same models with time-aware splits and find that Dataset 2's tuned XGBoost drops from **57.33% → ~52%** under chronological evaluation. The honest production-equivalent number is closer to 52% than 57%.